# CPBM — Full Pipeline Walkthrough

End-to-end execution of all four CPBM layers on a synthetic community of N=500,
with training, evaluation, diffusion simulation, and scaling aggregation.

**Repositories:** [OSF](https://osf.io/9ukxe/) | [https://github.com/ahmed-elsayed-99/CPBM-algorithm)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from cpbm.data.synthetic import SyntheticCommunity
from cpbm.core.signature import SignatureExtractor
from cpbm.core.diffusion import DiffusionLayer
from cpbm.core.stratum import StratumGradient
from cpbm.core.npi import NPICalculator
from cpbm.models.ensemble import CPBMEnsemble
from cpbm.evaluation.metrics import CPBMEvaluator
from cpbm.scaling.aggregator import RegionalAggregator
from cpbm.viz.plots import (
    plot_diffusion_curves, plot_stratum_adoption,
    plot_feature_importance, plot_pulse_clusters_pca,
    plot_full_dashboard
)

## Step 1 — Data Generation

In [ ]:
community    = SyntheticCommunity(n=500, seed=42).generate()
Phi          = community['phi_matrix']
labels       = community['labels']
strata       = community['strata']
transactions = community['transactions']

print('Stratum distribution and purchase rates:')
community['dataframe'].groupby('stratum')['label'].agg(['count','mean']).round(3)

## Step 2 — Layer 1: Pulse Signatures

In [ ]:
extractor = SignatureExtractor(window_days=90, k_range=(2,8), random_state=42)
Phi_ext   = extractor.fit_transform(transactions)

print(f'K* = {extractor.k_star_}')
extractor.get_cluster_profiles()

## Step 3 — Layer 2: Social Diffusion

In [ ]:
diffusion = DiffusionLayer(random_state=42)
t_h, a_h  = community['adoption_history']
diffusion.fit(t_h, a_h)
print(f"Fitted params: {diffusion.params_}")

diffusion.build_graph(n_nodes=len(Phi_ext))
seeds = np.where(Phi_ext[:, 0] >= np.percentile(Phi_ext[:, 0], 95))[0].tolist()
approx_proba = labels * 0.65 + 0.15

diff_results = diffusion.simulate(
    n_total=len(Phi_ext), seed_nodes=seeds,
    purchase_proba=approx_proba, steps=60
)
peak = diff_results.loc[diff_results['penetration_pct'].idxmax()]
print(f"Peak: day {int(peak['day'])}, penetration {peak['penetration_pct']:.1f}%")
plot_diffusion_curves(diff_results)
plt.show()

## Step 4 — Layer 3: Stratum Gradient and τ

In [ ]:
stratum_layer = StratumGradient()
stratum_layer.fit(community['stratum_adoption'])

summary = stratum_layer.summary()
print(summary.to_string(index=False))

tau_mean = float(summary['tau_days'].mean())
print(f"\nMean τ = {tau_mean:.2f} days")
interp = stratum_layer.tau_interpretation(tau_mean)
print(f"Category: {interp['category']} — {interp['strategy']}")

t_axis = np.linspace(0, 90, 91)
stratum_curves = {s: stratum_layer.predict_adoption(s, t_axis) for s in [1,2,3]}
plot_stratum_adoption(stratum_curves, t_axis, tau_dict=stratum_layer.tau_)
plt.show()

## Step 5 — NPI Calculation

In [ ]:
npi_calc = NPICalculator()
npi_val  = npi_calc.compute_from_dataframe(transactions, 'target')
print(f"NPI = {npi_val:.2f}")
print(f"Interpretation: {npi_calc.interpret(npi_val)}")

## Step 6 — Layer 4: Ensemble Training

In [ ]:
ensemble = CPBMEnsemble(random_state=42)
ensemble.fit(
    Phi=Phi_ext,
    labels=labels,
    tau=tau_mean,
    npi=npi_val,
    cluster_labels=extractor.cluster_labels_,
)
pd.DataFrame([ensemble.eval_results_]).T.rename(columns={0: 'value'}).round(4)

## Step 7 — Full Evaluation

In [ ]:
evaluator  = CPBMEvaluator()
all_proba  = ensemble.predict_proba(
    Phi_ext, tau=tau_mean, npi=npi_val,
    cluster_labels=extractor.cluster_labels_
)

result = evaluator.full_report(labels, all_proba, 'CPBM Ensemble')
print(f"AUC-ROC : {result['auc_roc']:.4f}")
print(f"AUC-PR  : {result['auc_pr']:.4f}")
print(f"F1      : {result['f1']:.4f}")

print('\nPer-stratum:')
print(evaluator.stratum_report(labels, all_proba, strata).to_string(index=False))

print('\nCalibration:')
print(evaluator.calibration_report(labels, all_proba, n_bins=8).to_string(index=False))

## Step 8 — Model Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_tab = np.hstack([
    Phi_ext,
    np.full((len(Phi_ext), 1), tau_mean),
    np.full((len(Phi_ext), 1), npi_val),
    extractor.cluster_labels_.reshape(-1, 1)
])

X_tr, X_te, y_tr, y_te = train_test_split(
    X_tab, labels, test_size=0.2, stratify=labels, random_state=42
)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

lr = LogisticRegression(max_iter=500).fit(X_tr_s, y_tr)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)

proba_te = ensemble.predict_proba(
    Phi_ext[X_tab.tolist().index(X_te[0].tolist()):] if False else Phi_ext,
    tau=tau_mean, npi=npi_val, cluster_labels=extractor.cluster_labels_
)

comparison = evaluator.compare_models(
    y_te,
    {
        'Logistic Regression': lr.predict_proba(X_te_s)[:, 1],
        'Random Forest'      : rf.predict_proba(X_te)[:, 1],
        'XGBoost only'       : ensemble.xgb.predict_proba(X_te),
    }
)
comparison

## Step 9 — Feature Importance

In [ ]:
imp_df = ensemble.xgb.get_feature_importance_df()
plot_feature_importance(imp_df)
plt.show()
print(imp_df.to_string(index=False))

## Step 10 — Scaling Demo

In [ ]:
agg = RegionalAggregator()
for i, (geo, n) in enumerate(zip(
    ['GEO_A','GEO_B','GEO_C'],
    [300, 450, 500]
)):
    c2   = SyntheticCommunity(n=n, seed=i+10).generate()
    dl2  = DiffusionLayer(random_state=i+10)
    dl2.fit(*c2['adoption_history'])
    sl2  = StratumGradient()
    sl2.fit(c2['stratum_adoption'])
    taus = list(sl2.tau_.values())
    npi2 = NPICalculator().compute_from_dataframe(c2['transactions'], 'target')
    agg.add_community(
        geo_code=geo, npi=npi2, D=dl2.params_['D'],
        tau_min=min(taus) if taus else 7.0,
        tau_max=max(taus) if taus else 21.0,
        population=n
    )

macro = {'cpi_growth': 0.087, 'unemployment': 0.112, 'gdp_growth': 0.034}
feat  = agg.aggregate(macro)
names = ['mean_NPI','std_NPI','mean_D','min_tau','tau_range',
         'cpi_growth','unemployment','gdp_growth']
pd.DataFrame({'feature': names, 'value': feat.round(5)})

## Step 11 — Full Dashboard

In [ ]:
stratum_proba_dict = {int(s): all_proba[strata == s] for s in np.unique(strata)}
plot_full_dashboard(
    diff_results, imp_df, Phi_ext,
    extractor.cluster_labels_,
    stratum_proba_dict, strata
)
plt.show()